In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF document
file_path = "employee-handbook.pdf"
loader = PyPDFLoader(file_path)

# Load all pages
docs = loader.load()

print(f"Loaded {len(docs)} pages from the PDF")
print(f"\nFirst page preview (first 500 characters):")
print(docs[0].page_content[:500])
print(f"\nMetadata: {docs[0].metadata}")

C:\Users\Training\AppData\Local\Temp\ipykernel_27228\4032902984.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(24, 0, 2725854725856), '/LastChar': 89, '/Widths': [222, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 444, 500, 0, 500, 444, 444, 0, 500, 278, 0, 444, 389, 611, 500, 500, 500, 0, 500, 444, 0, 500, 0, 0, 0, 444], '/Name': '/F0', '/BaseFont': '/BIOOHR+TradeGothic-BoldCondTwenty', '/FirstChar': 32, '/Encoding': IndirectObject(22, 0, 2725854725856), '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encoun

Loaded 35 pages from the PDF

First page preview (first 500 characters):
MODEL EMPLOYEE HANDBOOK 
FOR SMALL BUSINESS

Metadata: {'producer': 'QuarkXPress(tm) 6.5', 'creator': 'QuarkXPress(tm) 6.5', 'creationdate': '2005-11-16T16:29:38+00:00', 'moddate': '2017-03-07T10:21:06-05:00', 'xpressprivate': '%%DocumentProcessColors: Cyan Magenta Yellow Black\n%%DocumentCustomColors: (PANTONE 185 C)\n%%+ (PANTONE Blue 072 C)\n%%+ (PANTONE Warm Gray 9 U)\n%%+ (PANTONE 1797 U)\n%%CMYKCustomColor: 0 .91 .76 0 (PANTONE 185 C)\n%%+ 1 .88 0 .05 (PANTONE Blue 072 C)\n%%+ 0 .11 .2 .47 (PANTONE Warm Gray 9 U)\n%%+ 0 1 .99 .04 (PANTONE 1797 U)\n%%EndComments', 'source': 'employee-handbook.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}


In [3]:
# Informational example only – fixed-size chunking WITH overlap
def fixed_size_chunk(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # step back so boundaries overlap
    return chunks

sample = "Employee vacation policy allows 15 days per year. Sick leave is separate..."
print(fixed_size_chunk(sample, chunk_size=40, overlap=10))

['Employee vacation policy allows 15 days ', 's 15 days per year. Sick leave is separa', ' is separate...']


In [4]:
# Informational example only – separator hierarchy (concept behind RecursiveCharacterTextSplitter)
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

def recursive_split(text: str, chunk_size: int, separators: list[str]) -> list[str]:
    if not separators or len(text) <= chunk_size:
        return [text] if text else []
    sep = separators[0]
    parts = text.split(sep) if sep else list(text)
    chunks, current = [], ""
    for part in parts:
        candidate = (current + sep + part) if current else part
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current)
            if len(part) > chunk_size:
                chunks.extend(recursive_split(part, chunk_size, separators[1:]))
                current = ""
            else:
                current = part
    if current:
        chunks.append(current)
    return chunks

In [6]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
semantic_splitter = SemanticChunker(
    embed_model,
    breakpoint_threshold_type="percentile",  # split at similarity drops
    breakpoint_threshold_amount=90,
)
# semantic_chunks = semantic_splitter.split_text(long_document)

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # Maximum chunk size
    chunk_overlap=200,      # Overlap to maintain context
    add_start_index=True,   # Track position in original doc
    length_function=len,
)

# Split documents into chunks
all_splits = text_splitter.split_documents(docs)

print(f"Split {len(docs)} pages into {len(all_splits)} chunks")
print(f"\nExample chunk:")
print(all_splits[0].page_content)
print(f"\nChunk metadata: {all_splits[0].metadata}")

Split 35 pages into 81 chunks

Example chunk:
MODEL EMPLOYEE HANDBOOK 
FOR SMALL BUSINESS

Chunk metadata: {'producer': 'QuarkXPress(tm) 6.5', 'creator': 'QuarkXPress(tm) 6.5', 'creationdate': '2005-11-16T16:29:38+00:00', 'moddate': '2017-03-07T10:21:06-05:00', 'xpressprivate': '%%DocumentProcessColors: Cyan Magenta Yellow Black\n%%DocumentCustomColors: (PANTONE 185 C)\n%%+ (PANTONE Blue 072 C)\n%%+ (PANTONE Warm Gray 9 U)\n%%+ (PANTONE 1797 U)\n%%CMYKCustomColor: 0 .91 .76 0 (PANTONE 185 C)\n%%+ 1 .88 0 .05 (PANTONE Blue 072 C)\n%%+ 0 .11 .2 .47 (PANTONE Warm Gray 9 U)\n%%+ 0 1 .99 .04 (PANTONE 1797 U)\n%%EndComments', 'source': 'employee-handbook.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1', 'start_index': 0}


In [8]:
from langchain_openai import OpenAIEmbeddings

# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Let's create embeddings for sample texts to understand how they work
sample_texts = [
    "Employee vacation policy and time off",
    "Annual leave and holiday guidelines",
    "Company dress code requirements"
]

# Generate embeddings
sample_embeddings = [embeddings.embed_query(text) for text in sample_texts]

print(f"Each embedding has {len(sample_embeddings[0])} dimensions")
print(f"\nFirst 10 values of embedding 1: {sample_embeddings[0][:10]}")

# Calculate similarity between embeddings (using dot product as approximation)
import numpy as np

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

sim_1_2 = cosine_similarity(sample_embeddings[0], sample_embeddings[1])
sim_1_3 = cosine_similarity(sample_embeddings[0], sample_embeddings[2])

print(f"\nSimilarity between text 1 and 2 (both about time off): {sim_1_2:.4f}")
print(f"Similarity between text 1 and 3 (different topics): {sim_1_3:.4f}")
print("\n✅ Higher similarity = more related content!")

Each embedding has 1536 dimensions

First 10 values of embedding 1: [-0.03509521484375, 0.01137542724609375, 0.01983642578125, 0.0088348388671875, -0.002346038818359375, 0.02142333984375, -0.00839996337890625, 0.03668212890625, 0.033538818359375, 0.005405426025390625]

Similarity between text 1 and 2 (both about time off): 0.5644
Similarity between text 1 and 3 (different topics): 0.2859

✅ Higher similarity = more related content!


In [ ]:
import chromadb
#chroma run --path ./chroma_server_data --host localhost --port 8000  | in terminal to run instance at 8000
from langchain_community.vectorstores import Chroma

# Connect to the standalone Chroma server
chroma_client = chromadb.HttpClient(host="localhost", port=8000)

# Quick health check – should print a small integer (nanoseconds since epoch)
print(f"Chroma heartbeat: {chroma_client.heartbeat()}")
print(f"Existing collections: {chroma_client.list_collections()}")

# Create (or open) a collection and ingest the handbook chunks
vector_store = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    client=chroma_client,
    collection_name="employee_handbook",
)

print(f"✅ Connected to standalone Chroma at localhost:8000")
print(f"✅ Collection 'employee_handbook' now has {len(all_splits)} chunks")

Chroma heartbeat: 1790685728118762900
Existing collections: []
✅ Connected to standalone Chroma at localhost:8000
✅ Collection 'employee_handbook' now has 81 chunks


In [11]:
# Later sessions: connect without re-ingesting
vector_store = Chroma(
    client=chroma_client,
    collection_name="employee_handbook",
    embedding_function=embeddings,
)

In [12]:
# Simple similarity search
query = "What is the vacation policy?"

# Find top 3 most relevant documents
results = vector_store.similarity_search(query, k=3)

print(f"Query: '{query}'\n")
print(f"Found {len(results)} relevant documents:\n")

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(f"Content: {doc.page_content[:300]}...")
    print(f"Metadata: {doc.metadata}")
    print()

Query: 'What is the vacation policy?'

Found 3 relevant documents:

--- Result 1 ---
Content: Section 6 – Leave Policies  
6.1 Vacations 
The company provides, as a benefit, paid vacations for its eligible employees. Forward 
requests for time off in advance to a supervisor, who may approve or deny the request 
based on company resources. The company is flexible in approving time off when do...
Metadata: {'start_index': 0, 'creationdate': '2005-11-16T16:29:38+00:00', 'xpressprivate': '%%DocumentProcessColors: Cyan Magenta Yellow Black\n%%DocumentCustomColors: (PANTONE 185 C)\n%%+ (PANTONE Blue 072 C)\n%%+ (PANTONE Warm Gray 9 U)\n%%+ (PANTONE 1797 U)\n%%CMYKCustomColor: 0 .91 .76 0 (PANTONE 185 C)\n%%+ 1 .88 0 .05 (PANTONE Blue 072 C)\n%%+ 0 .11 .2 .47 (PANTONE Warm Gray 9 U)\n%%+ 0 1 .99 .04 (PANTONE 1797 U)\n%%EndComments', 'creator': 'QuarkXPress(tm) 6.5', 'source': 'employee-handbook.pdf', 'total_pages': 35, 'page': 15, 'moddate': '2017-03-07T10:21:06-05:00', 'page_label': '16', 'p

In [13]:
# Similarity search with scores
query = "How many sick days do employees get?"

# Get results with similarity scores
results_with_scores = vector_store.similarity_search_with_score(query, k=3)

print(f"Query: '{query}'\n")

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"--- Result {i} (Score: {score:.4f}) ---")
    print(f"Content: {doc.page_content[:200]}...")
    print()

Query: 'How many sick days do employees get?'

--- Result 1 (Score: 0.8242) ---
Content: person’s name] regarding the amount of (paid) sick leave provided each year.  Sick days 
may not be carried over into the next year. Abuse of this policy may result in 
disciplinary action.  
6.3 Fami...

--- Result 2 (Score: 0.8316) ---
Content: year, but cannot exceed 10 days of carryover. Vacation benefits do not accrue during 
any period of extended leave of absence.  
Generally, states do not require vacation benefits for employees, but m...

--- Result 3 (Score: 0.9491) ---
Content: Coverage extends for [enter number] days of disability. Employees must exhaust any 
sick leave benefits before being eligible for disability leave coverage.  
 
Disability benefits are calculated as [...



In [14]:
# Try more queries
queries = [
    "What are the working hours?",
    "Tell me about employee benefits",
    "What is the dress code policy?"
]

for query in queries:
    results = vector_store.similarity_search(query, k=1)
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    print(results[0].page_content[:300])
    print("...")


Query: What are the working hours?
Section 5 – Attendance Policies 
5.1 General Attendance 
The company maintains normal working hours of 8 a.m. to 5 p.m.   Hours may vary 
depending on work location and job responsibilities.  Supervisors will provide 
employees with their work schedule.  Should an employee have any questions 
regard
...

Query: Tell me about employee benefits
Section 10 - Employee Benefits 
 
This handbook contains descriptions of some  of our current employee benefits.  Many 
of the company’s benefit plans are described in more formal plan documents available 
from [enter authorized person’s name].  In the event of any inconsistencies between this 
hand
...

Query: What is the dress code policy?
Section 3 – Company Policies and Procedures 
3.1 Professional Conduct 
This company expects its employees to adhere to a standard of professional conduct 
and integrity. This ensures that the work environment is safe, comfortable and 
productive. Employees should be respectf

In [15]:
# Create a retriever from vector store
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Return top 3 results
)

# Test the retriever
query = "What are the company holidays?"
docs = retriever.invoke(query)

print(f"Query: '{query}'")
print(f"\nRetrieved {len(docs)} documents:\n")

for i, doc in enumerate(docs, 1):
    print(f"Document {i}:")
    print(doc.page_content[:250])
    print("...\n")

Query: 'What are the company holidays?'

Retrieved 3 documents:

Document 1:
all of his/her accrued vacation and sick leave.   
 
 
 
 
 
 
 
 
 
6.4 Holidays 
 
The company observes the following holidays: 
- New Year’s Day 
- Martin Luther King, Jr. Day 
- Memorial Day 
- Independence Day 
- Labor Day 
- Thanksgiving  
- Ch
...

Document 2:
terms and conditions as provided by the Family and Medical Leave Act.   
 
If your company has 50 or more employees, you are required to comply with 
the Family and Medical Leave Act. If you are covered, your manual should 
include a FMLA section. Fo
...

Document 3:
company. Include information regarding uniforms, safety protections such as 
steel toe shoes or hairnets, or other dress requirements. If the company provides
uniforms, consider including a caveat about lost uniform charges or laundry. 
3.3 Payday 
P
...



In [16]:
# MMR (Maximum Marginal Relevance) retriever
# This balances relevance with diversity

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,  # Fetch 10 candidates, then select 3 diverse ones
        "lambda_mult": 0.5  # 0=max diversity, 1=max relevance
    }
)

query = "What benefits do employees receive?"
docs = mmr_retriever.invoke(query)

print(f"Query: '{query}'")
print(f"\nMMR Retrieved documents (diverse + relevant):\n")

for i, doc in enumerate(docs, 1):
    print(f"Document {i}:")
    print(doc.page_content[:200])
    print("...\n")

Query: 'What benefits do employees receive?'

MMR Retrieved documents (diverse + relevant):

Document 1:
more detailed information on the benefits provided. 
 
 
 
 
 
 
10.1 Health Insurance 
 
The company makes group health benefits available to eligible employees and their 
family members. Eligible em
...

Document 2:
Companies may also consider instituting some of the following policies, 
depending on the nature of the company’s business and workforce: 
 
- Confidentiality 
- Conflict of Interest 
- Intellectual P
...

Document 3:
employee, and may include, for example, contributions to benefit plans.  Employees 
may contact Human Resources to obtain the necessary authorization forms for 
requesting additional deductions from t
...



In [17]:
# Batch retrieval - process multiple queries efficiently
questions = [
    "What is the remote work policy?",
    "How do I request time off?",
    "What is the probation period?"
]

# Batch invoke
batch_results = retriever.batch(questions)

for question, docs in zip(questions, batch_results):
    print(f"\nQuestion: {question}")
    print(f"Top result: {docs[0].page_content[:150]}...\n")


Question: What is the remote work policy?
Top result: Companies may also consider instituting some of the following policies, 
depending on the nature of the company’s business and workforce: 
 
- Confide...


Question: How do I request time off?
Top result: Section 6 – Leave Policies  
6.1 Vacations 
The company provides, as a benefit, paid vacations for its eligible employees. Forward 
requests for time ...


Question: What is the probation period?
Top result: 8.2 Procedures 
 
Disciplinary action is any one of a number of options used to correct unacceptable 
behavior or actions. Discipline may take the for...



In [18]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent  # LangChain 1.0 way
from langchain.tools import tool

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create a retrieval tool using the @tool decorator
@tool
def retrieve_employee_handbook(query: str):
    """
    Searches the employee handbook for information.
    Use this tool when you need to find specific information about:
    - Company policies
    - Employee benefits
    - Working hours and time off
    - Code of conduct
    - Any other employee-related information
    
    Args:
        query: The question or topic to search for
    
    Returns:
        Relevant information from the employee handbook
    """
    # Retrieve relevant documents
    retrieved_docs = vector_store.similarity_search(query, k=3)
    
    # Format and return the content for the LLM
    return "\n\n".join([
        f"Source (Page {doc.metadata.get('page', 'N/A')}):\n{doc.page_content}"
        for doc in retrieved_docs
    ])

# Create tools list
tools = [retrieve_employee_handbook]

# Create the agent using LangChain 1.0's create_agent
# This is simpler than the old create_tool_calling_agent + AgentExecutor approach
system_prompt = """You are a helpful HR assistant with access to the company employee handbook.

Use the retrieve_employee_handbook tool to search for information when answering questions about:
- Company policies and procedures
- Employee benefits and perks  
- Working hours, holidays, and time off
- Code of conduct and guidelines
- Any other employment-related topics

When you receive information from the tool, provide a clear and helpful answer based on that context.
If you can answer without the tool (like greetings), feel free to do so.
If the information isn't found, say so honestly."""

agent = create_agent(llm, tools, system_prompt=system_prompt)

print("✅ RAG Agent created successfully using LangChain 1.0!")

✅ RAG Agent created successfully using LangChain 1.0!


In [19]:
# Test the RAG agent
# In LangChain 1.0, we use stream() with messages format
question = "What is the vacation policy for employees?"

print("\n" + "="*60)
print("QUESTION:", question)
print("="*60)
print("\nAgent thinking process:")
print("-"*60)

# Stream the agent's execution
for event in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values"
):
    event["messages"][-1].pretty_print()

print("="*60)


QUESTION: What is the vacation policy for employees?

Agent thinking process:
------------------------------------------------------------
================================ Human Message =================================

What is the vacation policy for employees?
================================== Ai Message ==================================
Tool Calls:
  retrieve_employee_handbook (call_52V0ReoghC8zOnMC77u93ZgT)
 Call ID: call_52V0ReoghC8zOnMC77u93ZgT
  Args:
    query: vacation policy
================================= Tool Message =================================
Name: retrieve_employee_handbook

Source (Page 15):
Section 6 – Leave Policies  
6.1 Vacations 
The company provides, as a benefit, paid vacations for its eligible employees. Forward 
requests for time off in advance to a supervisor, who may approve or deny the request 
based on company resources. The company is flexible in approving time off when doing 
so would not interfere with company operations. Vacation days are gr

In [20]:
# Test with a question that needs multiple searches
complex_question = "What are the sick leave benefits and how do they compare to vacation days?"

print("\n" + "="*60)
print("COMPLEX QUESTION:", complex_question)
print("="*60)
print("\nAgent execution (may do multiple tool calls):")
print("-"*60)

for event in agent.stream(
    {"messages": [{"role": "user", "content": complex_question}]},
    stream_mode="values"
):
    event["messages"][-1].pretty_print()

print("="*60)


COMPLEX QUESTION: What are the sick leave benefits and how do they compare to vacation days?

Agent execution (may do multiple tool calls):
------------------------------------------------------------
================================ Human Message =================================

What are the sick leave benefits and how do they compare to vacation days?
================================== Ai Message ==================================
Tool Calls:
  retrieve_employee_handbook (call_vxMRxKQM1NZETKJgCGSL2d6f)
 Call ID: call_vxMRxKQM1NZETKJgCGSL2d6f
  Args:
    query: sick leave benefits
  retrieve_employee_handbook (call_LYFusoIS01eBvmYFvE8Hrv1m)
 Call ID: call_LYFusoIS01eBvmYFvE8Hrv1m
  Args:
    query: vacation days
================================= Tool Message =================================
Name: retrieve_employee_handbook

Source (Page 15):
year, but cannot exceed 10 days of carryover. Vacation benefits do not accrue during 
any period of extended leave of absence.  
Generally, s

In [21]:
# Test with a greeting (should NOT use the tool)
greeting = "Hello! How are you?"

print("\n" + "="*60)
print("GREETING:", greeting)
print("="*60)
print("\n👀 Watch: The agent should respond without using the retrieval tool!")
print("-"*60)

for event in agent.stream(
    {"messages": [{"role": "user", "content": greeting}]},
    stream_mode="values"
):
    event["messages"][-1].pretty_print()

print("="*60)
print("\n✅ Notice: The agent didn't use the retrieval tool for a greeting!")


GREETING: Hello! How are you?

👀 Watch: The agent should respond without using the retrieval tool!
------------------------------------------------------------
================================ Human Message =================================

Hello! How are you?
================================== Ai Message ==================================

Hello! I'm here and ready to assist you. How can I help you today?

✅ Notice: The agent didn't use the retrieval tool for a greeting!


In [22]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

# Create a dynamic prompt that injects retrieved context
@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    """Inject context from vector store into the system message."""
    # Get the last user message
    last_query = request.state["messages"][-1].content
    
    # Retrieve relevant documents
    retrieved_docs = vector_store.similarity_search(last_query, k=3)
    
    # Format the context
    docs_content = "\n\n---\n\n".join([
        f"Page {doc.metadata.get('page', 'N/A')}:\n{doc.page_content}"
        for doc in retrieved_docs
    ])
    
    # Create system message with context
    system_message = (
        "You are a helpful HR assistant. Use the following context from the employee handbook to answer the question.\n\n"
        "If the answer is not in the context, say \"I don't have that information in the employee handbook.\"\n"
        "Be specific and cite relevant policies when possible.\n\n"
        f"Context:\n{docs_content}"
    )
    
    return system_message

# Create the RAG pipeline using create_agent with middleware
rag_pipeline = create_agent(llm, tools=[], middleware=[prompt_with_context])

print("✅ RAG pipeline created successfully!")
print("\nHow it works:")
print("1. User asks a question")
print("2. Middleware retrieves relevant context from vector store")
print("3. Context is injected into system prompt")
print("4. LLM generates answer with context (single call)")

✅ RAG pipeline created successfully!

How it works:
1. User asks a question
2. Middleware retrieves relevant context from vector store
3. Context is injected into system prompt
4. LLM generates answer with context (single call)


In [23]:
# Test the RAG pipeline
question = "What are the working hours at the company?"

print("\n" + "="*60)
print("QUESTION:", question)
print("="*60)
print("\nResponse:")
print("-"*60)

for event in rag_pipeline.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values"
):
    event["messages"][-1].pretty_print()

print("="*60)


QUESTION: What are the working hours at the company?

Response:
------------------------------------------------------------
================================ Human Message =================================

What are the working hours at the company?
================================== Ai Message ==================================

The company maintains normal working hours of 8 a.m. to 5 p.m. However, hours may vary depending on work location and job responsibilities. Supervisors will provide employees with their specific work schedule. If an employee has any questions regarding their work schedule, they should contact their supervisor.


In [24]:
# Test with multiple questions
questions = [
    "How many paid holidays do employees get?",
    "What is the dress code policy?",
    "What benefits are offered to employees?"
]

print("\n" + "="*70)
print("TESTING MULTIPLE QUESTIONS")
print("="*70)

for q in questions:
    print(f"\nQ: {q}")
    print("-" * 60)
    
    # Get the response
    for event in rag_pipeline.stream(
        {"messages": [{"role": "user", "content": q}]},
        stream_mode="values"
    ):
        last_msg = event["messages"][-1]
        if hasattr(last_msg, 'content') and last_msg.content:
            print(f"A: {last_msg.content[:300]}...")
            break
    print()


TESTING MULTIPLE QUESTIONS

Q: How many paid holidays do employees get?
------------------------------------------------------------
A: How many paid holidays do employees get?...


Q: What is the dress code policy?
------------------------------------------------------------
A: What is the dress code policy?...


Q: What benefits are offered to employees?
------------------------------------------------------------
A: What benefits are offered to employees?...

